## Calculating the Units for Multi-family Homes across California

This notebook utilizes parcel data, Zillow data, and building data to calculate missing multi-family unit data. Residential parcels are subset to calculate building volume and create a regression that gives best approximations of multi-family home data. Single family home volume is additionally calculated. This unit data is utilized in calculating hosting capacity to assess where limited distributed energy resources exist across California.



In [1]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [2]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

### Parcel data

The analysis relies on parcels to calculate which buildings are assigned to which units. Some buildings intersect with more than one parcel which is kept in both calculations would duplicate the volume of thse homes and cause erroneous unit calculations. Hence a unique parcel number is required to ensure that the calculations accurate.

In [3]:
# load parcels 
parcels = gpd.read_parquet("/../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

In [4]:
# investigate duplicated parcel numbers
parcels[parcels['PARNO'].duplicated()]


,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry
2280,482-80-1-6,Alameda,None,None,None,172.340340,1242.904591,"MULTIPOLYGON Z (((-122.09155 37.59110 0.00000,..."
2413,941-2792-176,Alameda,None,None,None,1988.452430,6015.218730,"MULTIPOLYGON Z (((-121.91067 37.72566 0.00000,..."
2543,941-2792-176,Alameda,None,None,None,59.652862,92.678335,"MULTIPOLYGON Z (((-121.91158 37.72673 0.00000,..."
2711,946-4569-207-1,Alameda,None,None,None,75.019840,266.873353,"MULTIPOLYGON Z (((-121.84896 37.66600 0.00000,..."
5879,941-2792-176,Alameda,None,None,None,597.983026,1798.667011,"MULTIPOLYGON Z (((-121.91074 37.72763 0.00000,..."
...,...,...,...,...,...,...,...,...
11532333,common,Ventura,None,None,None,NaN,NaN,"POLYGON Z ((-119.06553 34.37366 0.00000, -119...."
11532334,common,Ventura,None,None,None,NaN,NaN,"POLYGON Z ((-119.05863 34.37424 0.00000, -119...."
11532335,common,Ventura,None,None,None,NaN,NaN,"POLYGON Z ((-118.83605 34.17332 0.00000, -118...."
11532336,common,Ventura,None,None,None,NaN,NaN,"POLYGON Z ((-118.83327 34.14923 0.00000, -118...."


The parcel number should have been unique to each parcel. As shown above there are 525,130 parcel numbers that are not unique. Instead we'll assume that each row is a different parcel and assign a unique id from the index. 

In [5]:
# make an id column in the parcel dataframe to use as the unique id
parcels['parcel_ID'] = parcels.index

In [6]:
parcels.head()

,PARNO,County,ADDRESS,CITY,ZIP,Shape_Length,Shape_Area,geometry,parcel_ID
0,48-6298-3-2,Alameda,None,None,None,103.683187,606.327284,"MULTIPOLYGON Z (((-122.12384 37.75571 0.00000,...",0
1,48-6313-23,Alameda,None,None,None,148.045063,1083.135315,"MULTIPOLYGON Z (((-122.12504 37.75429 0.00000,...",1
2,48-6313-87,Alameda,None,None,None,97.520632,557.505860,"MULTIPOLYGON Z (((-122.12635 37.75366 0.00000,...",2
3,48-6330-8-2,Alameda,None,None,None,171.271160,1602.682949,"MULTIPOLYGON Z (((-122.12243 37.75165 0.00000,...",3
4,48-6432-32,Alameda,None,None,None,140.208009,1078.107621,"MULTIPOLYGON Z (((-122.12829 37.76110 0.00000,...",4


### Zillow data

In [ ]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

# read in ca boundary to clip zillow
ca_boundary = gpd.read_file("/../../capstone/electrigrid/data/ca_state_boundary/CA_State.shp").to_crs(epsg=4326)

# clip Zillow to california 
zillow = zillow.clip(ca_boundary)

In [81]:
print(f"Number of Zillow points", zillow.shape[0])

Number of Zillow points 9071177


We expect to get the same number of homes at the end of unit regression calculation.

### Building Footprint data

Access: https://sat-io.earthengine.app/view/gba

In [9]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

# Unit-regression for multi-family homes and volume calculations for all homes

 Check that subsetting sums up to the total Zillow sum. 


In [82]:
# the analysis only requires the parcel_id and geometry  
parcels = parcels[['parcel_ID', 'geometry']]

## SUBSET ALL THE DATA TYPES
## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

# select the single family home data 
single_zillow = zillow[zillow['type'] == "Single"]

# select the condo data 
zillow_condos = zillow[(zillow['code'] == "RR106") & (zillow['code'] != 'Single' )]

# check to ensure all of the subsets are accurate, these all add up to the total of the zillow 
zillow_multi.shape[0] + single_zillow.shape[0] + zillow_condos.shape[0]

# set a zillow index column so we can track how much zillow data we lose
zillow_multi['zillow_index'] = zillow_multi.index

In [83]:
print(f"There are ",zillow_multi.shape[0], "multi-family zillow.")

There are  532343 multi-family zillow.


In [84]:
zillow_multi.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,geometry,zillow_index
7797783,Multi,1975.0,4.0,None,None,None,2.0,1609692.0,living,3319.0,8160547,06073021101,h352,SDGE,RI110,POINT (-116.57430 32.77842),7797783
7797430,Multi,1993.0,2.0,None,None,I,2.0,528750.0,living,2294.0,8160160,06073021302,h356,SDGE,RI110,POINT (-116.75519 32.74053),7797430
7797280,Multi,1989.0,8.0,None,None,None,3.0,1836000.0,living,6117.0,8159995,06073021302,h356,SDGE,RI110,POINT (-116.72715 32.74677),7797280
7797396,Multi,1978.0,4.0,None,None,I,2.0,660654.0,living,2409.0,8160125,06073021302,h356,SDGE,RI101,POINT (-116.75494 32.74932),7797396
7797376,Multi,1978.0,7.0,None,None,O,2.0,665856.0,living,2915.0,8160103,06073021302,h356,SDGE,RI110,POINT (-116.74799 32.75448),7797376


#### Valid parcel subset

Subset to parcels with multi-family homes.

In [85]:
## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains")
## output: each parcel where the a multi_zillow point is within it (one-to-many relationship)

# confirm that joining with Zillow decreases the number of parcels
assert len(valid_parcels) < len(parcels)

# drop units as we will add back summed units per parcel after
valid_parcels = valid_parcels.drop(['unit', 'index_right'], axis = 1)

# sum number of units per parcel
units_sum = valid_parcels.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
valid_parcels = valid_parcels.join(units_sum)
valid_parcels.rename(columns={"unit":"total_unit"}, inplace=True)
## for each parcel we should have the total number of units but this means that every time a parcel appears total units appear 

# drop duplicates based on parcel
valid_parcels = valid_parcels.drop_duplicates(subset = 'parcel_ID', keep = 'first')

In [86]:
valid_parcels.head()

,parcel_ID,geometry,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit
92,92,"MULTIPOLYGON Z (((-122.11883 37.75180 0.00000,...",Multi,1970.0,3.0,None,None,O,1447455.0,living,2621.0,135866,06001409900,262,PGE/SCE,RI101,123790,2.0
212,212,"MULTIPOLYGON Z (((-122.17728 37.72609 0.00000,...",Multi,1965.0,6.0,None,None,I,518707.0,living,3038.0,113661,06001409200,568,PGE/SCE,RI103,103006,4.0
231,231,"MULTIPOLYGON Z (((-122.12955 37.75105 0.00000,...",Multi,1955.0,7.0,None,None,O,1335766.0,living,4473.0,135372,06001410000,262,PGE/SCE,RI101,123305,2.0
300,300,"MULTIPOLYGON Z (((-122.25473 37.81432 0.00000,...",Multi,1966.0,36.0,None,None,None,8022880.0,living,24529.0,3702,06001403600,574,PGE/SCE,RI104,2970,26.0
318,318,"MULTIPOLYGON Z (((-122.28456 37.81261 0.00000,...",Multi,1905.0,4.0,None,None,None,270454.0,living,2625.0,162813,06001402400,468,PGE/SCE,RI101,148788,2.0


In [87]:
print(f"We lose ", len(zillow_multi['zillow_index'].unique()) - len(valid_parcels['zillow_index'].unique()), "homes when assigning to parcels.")

We lose  21008 homes when assigning to parcels.


This could be new developments. The parcel data being from 2014 does not capture all of the current development. These missing homes could be new developments. 
**Assign these homes to their nearest neighbors to calculate their volumes.**

In [89]:
# check the parcel_ID shape to keep track of losing any data
len(valid_parcels['parcel_ID'].unique())

544061

#### Buildings wuithin multi-family parcels

Assign buildings to parcels which already includes Zillow data. 

In [90]:
# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(valid_parcels, predicate="intersects")

# drop the unnecessary column 
valid_buildings = valid_buildings.drop(columns = 'index_right', axis = 1)

# confirm that joining with Zillow decreased the number of buildings
assert len(valid_buildings) < len(building)

In [91]:
valid_buildings.head()

,source,id,height,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit
18573,ms,UnitedStates_023010013_8142,4.508334,0.886133,USA,"{'xmax': -121.54639413033861, 'xmin': -121.546...","POLYGON ((-121.54641 40.08095, -121.54639 40.0...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0
18575,ms,UnitedStates_023010013_3773,3.229227,0.503842,USA,"{'xmax': -121.54630000377989, 'xmin': -121.546...","POLYGON ((-121.54630 40.08066, -121.54630 40.0...",480115,Multi,2007.0,1.0,None,None,None,200376.0,living,1004.0,525537,06007001601,460,PGE/SCE,RI000,453152,1.0
19630,ms,UnitedStates_023010012_33729,4.058523,0.654597,USA,"{'xmax': -121.87064520706676, 'xmin': -121.870...","POLYGON ((-121.87074 40.43491, -121.87065 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0
19631,ms,UnitedStates_023010012_16394,6.213455,0.869858,USA,"{'xmax': -121.8706356078727, 'xmin': -121.8707...","POLYGON ((-121.87072 40.43514, -121.87066 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0
19632,ms,UnitedStates_023010012_32724,2.698684,0.152872,USA,"{'xmax': -121.8708256701087, 'xmin': -121.8709...","POLYGON ((-121.87083 40.43542, -121.87090 40.4...",10631193,Multi,NaN,NaN,None,None,None,231599.0,None,NaN,10498457,06103000100,491,PGE/SCE,RI109,6596599,0.0


In [92]:
print(f"We lose ", len(valid_parcels['parcel_ID'].unique()) - len(valid_buildings['parcel_ID'].unique()), "parcels when assigning buildings")

We lose  7847 parcels when assigning buildings


**Could there potentially be some parcels with no buildings? Or have missing building data?"**

In [93]:
print(f"From raw zillow to buidlings we lose", len(zillow_multi['zillow_index'].unique()) - len(valid_buildings['zillow_index'].unique()) ,"zillow points")
print(f"From parcels to buildings we lose", len(valid_parcels['zillow_index'].unique()) - len(valid_buildings['zillow_index'].unique()), "zillow points")

From raw zillow to buidlings we lose 28534 zillow points
From parcels to buildings we lose 7526 zillow points


#### NOT CURRENTLY USED IN THE REST OF THE ANALYSIS: Clean building data 

Buildings can straddle more than one parcel. This causes the volume to be utilized in the regression more than once. To avoid this we assign the building to the parcel where more of the building area is.

**ISSUES WITH LOSING DATA**

In [94]:
## FOR BUILDINGS INTERSECTING MORE THAN ONE PARCEL CALCULATE WHICH PARCEL IT INTERSECTS MORE
# change the crs to a projected crs
building_zillow_parcels = valid_buildings.to_crs("EPSG:6933")
valid_parcels_proj = valid_parcels.to_crs("EPSG:6933").set_index('parcel_ID')

# to fix geometry problems from the crs change
building_zillow_parcels.geometry = building_zillow_parcels.geometry.buffer(0)
valid_parcels_proj.geometry = valid_parcels_proj.geometry.buffer(0)

# calculate intersection area for each building-parcel pair
building_zillow_parcels["intersection_area"] = (
    building_zillow_parcels.geometry.values
    .intersection(valid_parcels_proj.loc[building_zillow_parcels['parcel_ID']].geometry.values).area)

# keep only the parcel with the largest overlap per building
building_zillow_parcels = (
    building_zillow_parcels
    .sort_values("intersection_area", ascending=False)
    .loc[~building_zillow_parcels.index.duplicated(keep="first")]
    .drop(columns="intersection_area"))

In [49]:
building_zillow_parcels.head()

,source,id,height,var,region,bbox,geometry,parcel_ID,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,unit
1121611,osm,37060113,13.279422,13.296852,USA,"{'xmax': -117.56042480000002, 'xmin': -117.562...","POLYGON ((-11343156.647 4096570.487, -11343061...",7489961,Multi,1948.0,4.0,elec,none,None,282531.0,living,1280.0,7225085,06071012700,1167,PGE/SCE,RI101,4768836,0.0
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",8604977,Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7377635,146.0
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",8213299,Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7377635,146.0
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",8245888,Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7377635,146.0
6469300,osm,297478493,6.800139,6.194139,USA,"{'xmax': -117.298237, 'xmin': -117.300623, 'ym...","POLYGON ((-11317670.568 3992764.586, -11317764...",8545765,Multi,NaN,NaN,None,None,None,11993890.0,None,NaN,7720847,06073017702,h635,SDGE,RI109,7377635,146.0


In [95]:
print(f"We lose",len(valid_buildings['zillow_index'].unique()) - len(building_zillow_parcels['zillow_index'].unique()), "zillow points when trying to remove duplicates." )

We lose 138599 zillow points when trying to remove duplicates.


**This code is obviously not working. Leaving them in below for now.**

#### Calculate volume

**Using valid buildings since we lose so much data when trying to account for the duplicates**

The regression runs on the assumption that volume is linearly correlated with the number of units. First we calculate volume using the area of buildings. Then we run a regression to calculate the number of units where missing in the data.

In [96]:
# reproject data frame to crs with meters as units
building_m = valid_buildings.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('parcel_ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'parcel_ID')

# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'parcel_ID', keep = 'first')

# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]


/tmp/ipykernel_3146926/681976932.py:55: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_3146926/681976932.py:56: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


In [97]:
slope

0.0014975013389160214

In [98]:
intercept

6.960721342568584

#### Utilize the regression to estimate units where they're missing  

In [103]:
# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) | (building_m['total_unit'] == 0)]

# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('ID')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'parcel_ID')

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

# utilize the values from the regression to calculate units
missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

## multiple buildings in one parcel have duplicated units since they were computed on total volume
#  only keep the first observation per parcel
missing_outlier_units_pred = missing_outlier_units_pred.drop_duplicates(subset = 'parcel_ID', keep = 'first')

# combine multi-family homes data frames
multi_complete = pd.concat([building_units_clean, missing_outlier_units_pred]).to_crs(zillow.crs)

# fill the total_volume for those with the missing_total_volume_m3 and rename to total_volume_m3
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# fill the total_volume for those with the missing_total_volume_m3 
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# drop unnecessary columns 
multi_complete = multi_complete.drop(['residual', 'geometry', 'total_volume_m3'], axis = 1)

# rename to total_volume_m3
multi_complete = multi_complete.rename(columns = {'missing_total_volume_m3' : 'total_volume_m3'})

#### Create points from the parcel polygons

Single family homes and condos are points not polygons. For uniformity and hosting capacity calculation set the points as centroids to the parcels.


In [104]:
# select only the geometry and parcel id to use for the centroid calculation
parcels_res = valid_parcels[['parcel_ID', 'geometry']]

# join the parcel data on ID to get the parcel geometry
multi_by_parcel = parcels_res.merge(multi_complete, on = 'parcel_ID')

# set the geometry to the parcel geometry
multi_by_parcel = multi_by_parcel.set_geometry('geometry')

# change the crs 
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# rename the centroid to geometry
multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

In [105]:
multi_by_parcel_processed.head()

,parcel_ID,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,zillow_index,total_unit,area_m2,volume_m3,total_volume_m3,geometry
0,92,osm,470801979,11.308458,1.151844,USA,"{'xmax': -122.1188566, 'xmin': -122.1191448, '...",Multi,1970.0,3.0,None,None,O,1447455.0,living,2621.0,135866,06001409900,262,PGE/SCE,RI101,123790,2.0,305.823964,3458.397556,6598.486312,POINT (-122.11915 37.75188)
1,212,osm,467433407,3.687969,0.562078,USA,"{'xmax': -122.1772724, 'xmin': -122.1774869999...",Multi,1965.0,6.0,None,None,I,518707.0,living,3038.0,113661,06001409200,568,PGE/SCE,RI103,103006,4.0,180.790440,666.749620,1476.620600,POINT (-122.17738 37.72601)
2,231,osm,471260067,12.743473,1.726028,USA,"{'xmax': -122.12957120000002, 'xmin': -122.129...",Multi,1955.0,7.0,None,None,O,1335766.0,living,4473.0,135372,06001410000,262,PGE/SCE,RI101,123305,2.0,452.441520,5765.676322,5818.222802,POINT (-122.12977 37.75103)
3,300,osm,309469329,7.206610,0.512426,USA,"{'xmax': -122.2546441, 'xmin': -122.2548513, '...",Multi,1966.0,36.0,None,None,None,8022880.0,living,24529.0,3702,06001403600,574,PGE/SCE,RI104,2970,26.0,176.758925,1273.832585,7779.240336,POINT (-122.25494 37.81419)
4,318,osm,473407799,5.020492,0.822397,USA,"{'xmax': -122.2846471, 'xmin': -122.2848646, '...",Multi,1905.0,4.0,None,None,None,270454.0,living,2625.0,162813,06001402400,468,PGE/SCE,RI101,148788,2.0,165.396340,830.370936,1936.266938,POINT (-122.28473 37.81260)


In [106]:
print(f"We lose ",zillow_multi.shape[0] - len(multi_by_parcel_processed['zillow_index'].unique()), "multi-family zillow points.")

We lose  33927 multi-family zillow points.


In [ ]:
# takes a really long time 
multi_by_parcel_processed.to_parquet("multi_zillow_w_units.parquet")